In [1]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO

In [2]:
PROJECT_PATH = Path(
    r"D:\master\Master_Drone_Detection"
)

DATASET_PATH = (
    PROJECT_PATH
    / "02_datasets"
    / "DUT_Anti_UAV"
)

# أثناء تطوير EXP003 نستخدم VAL
EVAL_SPLIT = "val"

IMAGES_DIR = (
    DATASET_PATH
    / "images"
    / EVAL_SPLIT
)

LABELS_DIR = (
    DATASET_PATH
    / "labels"
    / EVAL_SPLIT
)

EXP2_BEST_MODEL = (
    PROJECT_PATH
    / "notebooks"
    / "runs"
    / "04_experiments"
    / "EXP002_YOLOv8s_high_resolution"
    / "YOLOv8s_960-3"
    / "weights"
    / "best.pt"
)

RESULTS_DIR = (
    PROJECT_PATH
    / "06_results"
    / "EXP002_object_level_error_analysis"
    / EVAL_SPLIT
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model:", EXP2_BEST_MODEL)
print("Images:", IMAGES_DIR)
print("Labels:", LABELS_DIR)
print("Results:", RESULTS_DIR)

assert EXP2_BEST_MODEL.exists()
assert IMAGES_DIR.exists()
assert LABELS_DIR.exists()

Model: D:\master\Master_Drone_Detection\notebooks\runs\04_experiments\EXP002_YOLOv8s_high_resolution\YOLOv8s_960-3\weights\best.pt
Images: D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val
Labels: D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\labels\val
Results: D:\master\Master_Drone_Detection\06_results\EXP002_object_level_error_analysis\val


In [3]:
IMGSZ = 960

CONF_THRESHOLD = 0.25

MATCH_IOU_THRESHOLD = 0.50

NMS_IOU_THRESHOLD = 0.70

BATCH_SIZE = 8

In [4]:
model_exp2 = YOLO(
    str(EXP2_BEST_MODEL)
)

image_extensions = [
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.bmp"
]

image_paths = []

for ext in image_extensions:
    image_paths.extend(
        IMAGES_DIR.glob(ext)
    )

image_paths = sorted(image_paths)

print(
    "Number of evaluation images:",
    len(image_paths)
)

Number of evaluation images: 2600


In [5]:
def calculate_iou(box1, box2):

    x1 = max(
        box1[0],
        box2[0]
    )

    y1 = max(
        box1[1],
        box2[1]
    )

    x2 = min(
        box1[2],
        box2[2]
    )

    y2 = min(
        box1[3],
        box2[3]
    )


    intersection_width = max(
        0,
        x2 - x1
    )

    intersection_height = max(
        0,
        y2 - y1
    )


    intersection = (
        intersection_width
        *
        intersection_height
    )


    box1_area = (
        max(0, box1[2] - box1[0])
        *
        max(0, box1[3] - box1[1])
    )


    box2_area = (
        max(0, box2[2] - box2[0])
        *
        max(0, box2[3] - box2[1])
    )


    union = (
        box1_area
        +
        box2_area
        -
        intersection
    )


    if union == 0:
        return 0.0


    return intersection / union

In [6]:
def classify_uav_size(width, height):

    if width < 32 and height < 32:
        return "Tiny"

    elif width < 96 and height < 96:
        return "Small"

    elif width < 256 and height < 256:
        return "Medium"

    else:
        return "Large"

In [7]:
def get_ground_truth(image_path):

    label_path = (
        LABELS_DIR
        / f"{image_path.stem}.txt"
    )


    with Image.open(image_path) as image:

        image_width, image_height = (
            image.size
        )


    gt_objects = []


    if not label_path.exists():

        return gt_objects


    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        lines = f.readlines()


    for gt_id, line in enumerate(lines):

        values = line.strip().split()

        if len(values) != 5:
            continue


        cls, xc, yc, bw, bh = map(
            float,
            values
        )


        x1 = (
            xc - bw / 2
        ) * image_width

        y1 = (
            yc - bh / 2
        ) * image_height

        x2 = (
            xc + bw / 2
        ) * image_width

        y2 = (
            yc + bh / 2
        ) * image_height


        object_width = (
            x2 - x1
        )

        object_height = (
            y2 - y1
        )


        size_category = classify_uav_size(
            object_width,
            object_height
        )


        gt_objects.append({

            "gt_id": gt_id,

            "class": int(cls),

            "box": [
                x1,
                y1,
                x2,
                y2
            ],

            "width": object_width,

            "height": object_height,

            "size": size_category
        })


    return gt_objects

In [8]:
sample_image = image_paths[0]

print(
    "Image:",
    sample_image.name
)

sample_gt = get_ground_truth(
    sample_image
)

sample_gt

Image: 00001.jpg


[{'gt_id': 0,
  'class': 0,
  'box': [257.999701, 52.000049999999995, 369.999849, 121.99995],
  'width': 112.00014799999997,
  'height': 69.9999,
  'size': 'Medium'}]

In [9]:
prediction_records = []

predictions_by_image = defaultdict(list)


for start in range(
    0,
    len(image_paths),
    BATCH_SIZE
):

    batch_paths = image_paths[
        start:start + BATCH_SIZE
    ]


    results = model_exp2.predict(

        source=[
            str(path)
            for path in batch_paths
        ],

        imgsz=IMGSZ,

        conf=CONF_THRESHOLD,

        iou=NMS_IOU_THRESHOLD,

        device=0,

        verbose=False
    )


    for image_path, result in zip(
        batch_paths,
        results
    ):

        image_name = image_path.stem


        for pred_id, box in enumerate(
            result.boxes
        ):

            xyxy = (
                box.xyxy[0]
                .cpu()
                .numpy()
            )


            prediction = {

                "image": image_name,

                "pred_id": pred_id,

                "class": int(
                    box.cls[0]
                ),

                "confidence": float(
                    box.conf[0]
                ),

                "box": [
                    float(xyxy[0]),
                    float(xyxy[1]),
                    float(xyxy[2]),
                    float(xyxy[3])
                ]
            }


            predictions_by_image[
                image_name
            ].append(
                prediction
            )


            prediction_records.append(
                prediction
            )


    print(
        f"Processed "
        f"{min(start + BATCH_SIZE, len(image_paths))}"
        f"/{len(image_paths)}"
    )

Processed 8/2600
Processed 16/2600
Processed 24/2600
Processed 32/2600
Processed 40/2600
Processed 48/2600
Processed 56/2600
Processed 64/2600
Processed 72/2600
Processed 80/2600
Processed 88/2600
Processed 96/2600
Processed 104/2600
Processed 112/2600
Processed 120/2600
Processed 128/2600
Processed 136/2600
Processed 144/2600
Processed 152/2600
Processed 160/2600
Processed 168/2600
Processed 176/2600
Processed 184/2600
Processed 192/2600
Processed 200/2600
Processed 208/2600
Processed 216/2600
Processed 224/2600
Processed 232/2600
Processed 240/2600
Processed 248/2600
Processed 256/2600
Processed 264/2600
Processed 272/2600
Processed 280/2600
Processed 288/2600
Processed 296/2600
Processed 304/2600
Processed 312/2600
Processed 320/2600
Processed 328/2600
Processed 336/2600
Processed 344/2600
Processed 352/2600
Processed 360/2600
Processed 368/2600
Processed 376/2600
Processed 384/2600
Processed 392/2600
Processed 400/2600
Processed 408/2600
Processed 416/2600
Processed 424/2600
Proces

In [10]:
print(
    "Total predictions:",
    len(prediction_records)
)

Total predictions: 2623


In [11]:
def match_image_objects(
    gt_objects,
    predictions,
    iou_threshold=0.5
):

    tp_records = []
    fp_records = []
    fn_records = []


    # الأعلى confidence أولاً
    predictions = sorted(

        predictions,

        key=lambda x: x["confidence"],

        reverse=True
    )


    matched_gt_ids = set()


    for pred in predictions:

        pred_box = pred["box"]


        # IoU with ALL GT objects
        all_ious = []

        for gt in gt_objects:

            iou = calculate_iou(
                pred_box,
                gt["box"]
            )

            all_ious.append(
                (
                    gt["gt_id"],
                    iou
                )
            )


        if len(all_ious) > 0:

            best_gt_any_id, best_iou_any = max(

                all_ious,

                key=lambda x: x[1]
            )

        else:

            best_gt_any_id = None
            best_iou_any = 0.0


        # Find best UNMATCHED GT
        unmatched_candidates = [

            (gt_id, iou)

            for gt_id, iou
            in all_ious

            if gt_id not in matched_gt_ids
        ]


        if len(unmatched_candidates) > 0:

            best_gt_id, best_iou = max(

                unmatched_candidates,

                key=lambda x: x[1]
            )

        else:

            best_gt_id = None
            best_iou = 0.0


        # -------------------------
        # TRUE POSITIVE
        # -------------------------

        if (
            best_gt_id is not None
            and
            best_iou >= iou_threshold
        ):

            matched_gt_ids.add(
                best_gt_id
            )


            matched_gt = next(

                gt for gt in gt_objects

                if gt["gt_id"]
                == best_gt_id
            )


            tp_records.append({

                "pred_id":
                    pred["pred_id"],

                "gt_id":
                    best_gt_id,

                "confidence":
                    pred["confidence"],

                "iou":
                    best_iou,

                "gt_width":
                    matched_gt["width"],

                "gt_height":
                    matched_gt["height"],

                "gt_size":
                    matched_gt["size"]
            })


        # -------------------------
        # FALSE POSITIVE
        # -------------------------

        else:

            # No overlap at all
            if best_iou_any == 0:

                fp_type = "Background"


            # overlaps a GT strongly,
            # but GT was already matched
            elif best_iou_any >= iou_threshold:

                fp_type = "Duplicate"


            # overlap exists,
            # but localization is poor
            else:

                fp_type = "Localization"


            fp_records.append({

                "pred_id":
                    pred["pred_id"],

                "confidence":
                    pred["confidence"],

                "best_iou":
                    best_iou_any,

                "fp_type":
                    fp_type
            })


    # -----------------------------
    # FALSE NEGATIVES
    # -----------------------------

    for gt in gt_objects:

        if gt["gt_id"] not in matched_gt_ids:

            fn_records.append({

                "gt_id":
                    gt["gt_id"],

                "gt_width":
                    gt["width"],

                "gt_height":
                    gt["height"],

                "gt_size":
                    gt["size"]
            })


    return (
        tp_records,
        fp_records,
        fn_records
    )

In [12]:
all_tp = []
all_fp = []
all_fn = []
all_gt = []


for index, image_path in enumerate(
    image_paths
):

    image_name = image_path.stem


    gt_objects = get_ground_truth(
        image_path
    )


    predictions = predictions_by_image.get(
        image_name,
        []
    )


    tp, fp, fn = match_image_objects(

        gt_objects,

        predictions,

        iou_threshold=MATCH_IOU_THRESHOLD
    )


    # Add image name
    for row in tp:

        row["image"] = image_name

        all_tp.append(row)


    for row in fp:

        row["image"] = image_name

        all_fp.append(row)


    for row in fn:

        row["image"] = image_name

        all_fn.append(row)


    for gt in gt_objects:

        all_gt.append({

            "image":
                image_name,

            "gt_id":
                gt["gt_id"],

            "width":
                gt["width"],

            "height":
                gt["height"],

            "size":
                gt["size"]
        })


    if (
        index + 1
    ) % 250 == 0:

        print(
            f"Matched "
            f"{index + 1}"
            f"/{len(image_paths)}"
        )

Matched 250/2600
Matched 500/2600
Matched 750/2600
Matched 1000/2600
Matched 1250/2600
Matched 1500/2600
Matched 1750/2600
Matched 2000/2600
Matched 2250/2600
Matched 2500/2600


In [13]:
tp_df = pd.DataFrame(
    all_tp
)

fp_df = pd.DataFrame(
    all_fp
)

fn_df = pd.DataFrame(
    all_fn
)

gt_df = pd.DataFrame(
    all_gt
)

In [14]:
print(
    "TP:",
    len(tp_df)
)

print(
    "FP:",
    len(fp_df)
)

print(
    "FN:",
    len(fn_df)
)

print(
    "GT objects:",
    len(gt_df)
)

TP: 2433
FP: 190
FN: 188
GT objects: 2621


In [15]:
TP = len(tp_df)
FP = len(fp_df)
FN = len(fn_df)


precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0
)


recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)


f1 = (

    2
    * precision
    * recall
    /
    (precision + recall)

    if (
        precision + recall
    ) > 0

    else 0
)


print(
    f"TP        : {TP}"
)

print(
    f"FP        : {FP}"
)

print(
    f"FN        : {FN}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1 Score  : {f1:.4f}"
)


TP        : 2433
FP        : 190
FN        : 188
Precision : 0.9276
Recall    : 0.9283
F1 Score  : 0.9279


In [16]:
total_gt = len(
    gt_df
)

total_predictions = len(
    prediction_records
)


print(
    "Total GT:",
    total_gt
)

print(
    "TP + FN:",
    TP + FN
)


print(
    "\nTotal Predictions:",
    total_predictions
)

print(
    "TP + FP:",
    TP + FP
)

Total GT: 2621
TP + FN: 2621

Total Predictions: 2623
TP + FP: 2623


In [17]:
fp_summary = (
    fp_df["fp_type"]
    .value_counts()
)

fp_summary

fp_type
Background      122
Localization     46
Duplicate        22
Name: count, dtype: int64

In [18]:
fn_size_summary = (
    fn_df["gt_size"]
    .value_counts()
    .reindex(
        [
            "Tiny",
            "Small",
            "Medium",
            "Large"
        ],
        fill_value=0
    )
)

fn_size_summary

gt_size
Tiny      95
Small     79
Medium    11
Large      3
Name: count, dtype: int64

In [19]:
fn_size_percentage = (
    fn_size_summary
    /
    len(fn_df)
    *
    100
)

fn_size_percentage

gt_size
Tiny      50.531915
Small     42.021277
Medium     5.851064
Large      1.595745
Name: count, dtype: float64

In [20]:
gt_size_counts = (
    gt_df["size"]
    .value_counts()
)


tp_size_counts = (
    tp_df["gt_size"]
    .value_counts()
)


size_results = []


for size in [
    "Tiny",
    "Small",
    "Medium",
    "Large"
]:

    total_gt_size = (
        gt_size_counts.get(
            size,
            0
        )
    )


    tp_size = (
        tp_size_counts.get(
            size,
            0
        )
    )


    fn_size = (
        total_gt_size
        -
        tp_size
    )


    recall_size = (

        tp_size
        /
        total_gt_size

        if total_gt_size > 0

        else 0
    )


    size_results.append({

        "size":
            size,

        "GT":
            total_gt_size,

        "TP":
            tp_size,

        "FN":
            fn_size,

        "Recall":
            recall_size
    })


size_analysis_df = pd.DataFrame(
    size_results
)

size_analysis_df

,size,GT,TP,FN,Recall
0,Tiny,720,625,95,0.868056
1,Small,1427,1348,79,0.944639
2,Medium,315,304,11,0.965079
3,Large,159,156,3,0.981132


In [22]:
tp_df.to_csv(
    RESULTS_DIR / "true_positives.csv",
    index=False
)

fp_df.to_csv(
    RESULTS_DIR / "false_positives.csv",
    index=False
)

fn_df.to_csv(
    RESULTS_DIR / "false_negatives.csv",
    index=False
)

gt_df.to_csv(
    RESULTS_DIR / "ground_truth_objects.csv",
    index=False
)

size_analysis_df.to_csv(
    RESULTS_DIR / "recall_by_object_size.csv",
    index=False
)

In [23]:
summary_df = pd.DataFrame({

    "Metric": [
        "TP",
        "FP",
        "FN",
        "Precision",
        "Recall",
        "F1"
    ],

    "Value": [
        TP,
        FP,
        FN,
        precision,
        recall,
        f1
    ]
})


summary_df.to_csv(
    RESULTS_DIR / "summary.csv",
    index=False
)

summary_df

,Metric,Value
0,TP,2433.000000
1,FP,190.000000
2,FN,188.000000
3,Precision,0.927564
4,Recall,0.928272
5,F1,0.927918


In [24]:
size_order = [
    "Tiny",
    "Small",
    "Medium",
    "Large"
]

fn_size_summary = (
    fn_df["gt_size"]
    .value_counts()
    .reindex(
        size_order,
        fill_value=0
    )
)

fn_size_percentage = (
    fn_size_summary
    / len(fn_df)
    * 100
)


fn_analysis_df = pd.DataFrame({
    
    "FN_Count":
        fn_size_summary,
    
    "FN_Percentage":
        fn_size_percentage
})


fn_analysis_df

,FN_Count,FN_Percentage
gt_size,,
Tiny,95,50.531915
Small,79,42.021277
Medium,11,5.851064
Large,3,1.595745


In [25]:
gt_size_counts = (
    gt_df["size"]
    .value_counts()
    .reindex(
        size_order,
        fill_value=0
    )
)


tp_size_counts = (
    tp_df["gt_size"]
    .value_counts()
    .reindex(
        size_order,
        fill_value=0
    )
)


size_results = []


for size in size_order:

    total_gt = int(
        gt_size_counts[size]
    )

    tp_count = int(
        tp_size_counts[size]
    )

    fn_count = (
        total_gt
        -
        tp_count
    )

    recall = (
        tp_count / total_gt
        if total_gt > 0
        else 0
    )


    size_results.append({
        
        "Size":
            size,
        
        "GT":
            total_gt,
        
        "TP":
            tp_count,
        
        "FN":
            fn_count,
        
        "Recall":
            recall
    })


size_analysis_df = pd.DataFrame(
    size_results
)

size_analysis_df

,Size,GT,TP,FN,Recall
0,Tiny,720,625,95,0.868056
1,Small,1427,1348,79,0.944639
2,Medium,315,304,11,0.965079
3,Large,159,156,3,0.981132


In [26]:
size_analysis_df[
    "Recall_Percentage"
] = (
    size_analysis_df["Recall"]
    * 100
)

size_analysis_df

,Size,GT,TP,FN,Recall,Recall_Percentage
0,Tiny,720,625,95,0.868056,86.805556
1,Small,1427,1348,79,0.944639,94.463910
2,Medium,315,304,11,0.965079,96.507937
3,Large,159,156,3,0.981132,98.113208


In [27]:
fp_type_summary = (
    fp_df["fp_type"]
    .value_counts()
)

fp_type_summary

fp_type
Background      122
Localization     46
Duplicate        22
Name: count, dtype: int64

In [28]:
fp_type_percentage = (
    fp_type_summary
    / len(fp_df)
    * 100
)

fp_analysis_df = pd.DataFrame({
    
    "FP_Count":
        fp_type_summary,
    
    "Percentage":
        fp_type_percentage
})

fp_analysis_df

,FP_Count,Percentage
fp_type,,
Background,122,64.210526
Localization,46,24.210526
Duplicate,22,11.578947
